In [1]:
!pip install qdrant_client groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 7.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.4 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import os
from dotenv import dotenv_values
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import PointStruct, Document
from groq import Groq
import uuid
from google import genai

In [ ]:
# Load environment variables
config = dotenv_values(".env")
HF_TOKEN  = config.get("HF_TOKEN")
QDRANT_CLOUD_API_KEY = config.get("QDRANT_CLOUD_API_KEY")
QDRANT_CLOUD_ENDPOINT = config.get("QDRANT_CLOUD_ENDPOINT")
GROQ_API_KEY  = config["GROQ_API_KEY"]

# Make HuggingFace token available to the transformers library
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# Add .env Varaible Here If you are Using Google Colab



In [5]:
# Hugging Face Authentification
from huggingface_hub import login
login(token=HF_TOKEN)

In [6]:
collection_name = "Morrocan_Chat_Culture"
# Connection With QDrant Cloud
client_qdrant = QdrantClient(
    url=QDRANT_CLOUD_ENDPOINT,
    api_key=QDRANT_CLOUD_API_KEY,
    cloud_inference=True
)

In [7]:
# Collection Creation - Collection That Support Hybrid Search
client_qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
                    distance=models.Distance.COSINE,
                    size=384,
        ),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

/tmp/ipykernel_5025/620268484.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client_qdrant.recreate_collection(


True

In [18]:
# client_qdrant.delete_collection(collection_name=collection_name)

In [8]:
# Connection Wth Groq
client_groq = Groq(api_key=GROQ_API_KEY)

In [9]:
# Load Dataset From Hugging Face
dataset = load_dataset(
    "atlasia/Atlaset",
    split="train",
    streaming=True # we do not download this dataset we create object refrence o dataset
).select_columns(["text", "word_count"])

README.md:   0%|          | 0.00/7.61k [00:00<?, ?B/s]

In [10]:
# Add Doocuments (group of tokens) to Cloud
def add_documents_to_qdrant(documents):
    client_qdrant.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=uuid.uuid4().hex,
                vector={
                    "dense": models.Document(
                        text=doc,
                        model="sentence-transformers/all-MiniLM-L6-v2",
                    ),
                    "sparse": models.Document(
                        text=doc,
                        model="Qdrant/bm25",
                    ),
                },
                payload={"text": doc},
            )
            for doc in documents
        ]
    )

In [11]:
# Dense Search
def dense_search(query: str) -> list[models.ScoredPoint]:
    response = client_qdrant.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="sentence-transformers/all-MiniLM-L6-v2",
        ),
        using="dense",
        limit=3,
    )
    for res in response.points:
        print(res.payload)
dense_search("تكلم شوية على مغرب")

In [12]:
# Sparse Search
def sparse_search(query: str) -> list[models.ScoredPoint]:
    response = client_qdrant.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="Qdrant/bm25",
        ),
        using="sparse",
        limit=3,
    )
    for res in response.points:
        print(res.payload)
sparse_search("تكلم شوية على مغرب")

In [22]:
# Add Chunks Into Qdrant Cloud
def add_doc_qdrant_cloud():
    BATCH_SIZE = 10
    total_tokens = 100000
    counter = 0
    start_id = 5550
    sentences = []
    for row in dataset:
        # print(row)
        if total_tokens <= start_id:
            break
        if (row["word_count"] >= 50):
            sentences.append(row["text"])
            counter += 1
        else: continue
        if (BATCH_SIZE <= counter):
            # Push The sentences into QDrant VDB
            add_documents_to_qdrant(sentences)
            sentences = []
            counter = 0
            start_id += BATCH_SIZE
    return start_id

start_id = add_doc_qdrant_cloud()
print(f"Total points upserted: {start_id}")

ResponseHandlingException: The read operation timed out

In [ ]:
# Hybrid Search with Reciprocal Rank Fusion
'''
    Qdrant Combine Keyword Search with Semantic Search
    - Step1: Keyword Search to Get Relevent Chunks
    - Step2: Semantic Search to Get Also Relevent Chunks
    - Fusion of Two Relevent Chunks: By USING RRF = Reciprocal Rank Fusion
'''

def hybrid_search(query: str) -> list[models.ScoredPoint]:
    response = client_qdrant.query_points(
        collection_name=collection_name,
        prefetch=[
            # keyWord Search - Sparse Vector
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=3,
            ),
            # Semantic Search - Dense Vector
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=3,
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=3,
    )
    tokens = []
    for res in response.points:
        tokens.append(res.payload["text"])
    return tokens


In [ ]:
# Check Similarity Between Hype Doc And Chunks in VDB
def get_relevent_chunks(hyde_documents):
    return hybrid_search(hyde_documents)

In [84]:
# Get hypothetical Embedding Documents

def get_llm_documents(question):
    """Generate a short hypothetical documentation passage for `question`."""
    
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": (
                    "نتا مساعد كيعطي معلومات مفيدة. جاوب بالدارجة المغربية اللي ساهلة ومفهومة. "
                    "عطي جواب واضح ومختصر بلا إطالة." 
                ),
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0.7,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = [chunk.choices[0].delta.content for chunk in completion]
    res = [s for s in res if s]
    return "".join(res)


In [85]:
def generate_response(question, context):
    sys_prompt = f"""
        أنت مساعد متخصص في الإجابة على الأسئلة باللغة العربية، بما فيها الدارجة المغربية.
        استخدم السياق المقدم للإجابة على السؤال بدقة.
        
        التعليمات:
        - أجب دائماً بنفس لغة السؤال (الدارجة المغربية أو العربية الفصحى)
        - إذا كان السؤال بالدارجة المغربية، أجب بالدارجة المغربية
        - إذا ما عرفتيش الجواب، قول 'ما عرفتش'
        - استخدم المعلومات الموجودة في السياق فقط
        - كون مختصر وواضح في جوابك
        - إذا كان السياق لا يحتوي على معلومات كافية، قول 'المعلومات ناقصة فالسياق'

        السياق:
        {context}
    """
    
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0.3,  # Lower = more factual, less hallucination
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = []
    for chunk in completion:
        res.append(chunk.choices[0].delta.content)

    res = [s for s in res if s]
    res = "".join(res)
    return res

In [88]:
# Define Question
question = "علاش تفاوض المغرب مع فرنسا؟"

# Get hypothetical Embedding Documents
hyde_documents = get_llm_documents(question=question)
print(hyde_documents)

# Get Relevent Chunks From Qdart
relevent_chunks = get_relevent_chunks(hyde_documents=hyde_documents)
print("relevent_chunks: ", relevent_chunks)

# Get Response
context = "".join(relevent_chunks)
response = generate_response(question=question, context=context)
print(response)

مرة في الزمن، كان المغرب مصدراً للثروات، لكن بسبب التوسع الفرنسي في أفريقيا، بدأوا يريدون ان يتوسعوا في المغرب أيضاً. فبدأت المفاوضات بين المغرب وفرنسا. 

المفاوضات كانت لمنح فرنسا الحق في إدارة بعض المناطق في المغرب، وبدأت من سنة 1904. ونتجت من هذه المفاوضات معاهدة فيس-مدينة في سنة 1912، التي كانت تنص على أن المغرب يعين حاكماً فرنسياً على البلاد. 

كان الغرض من هذه المعاهدة هو ان فرنسا تدير شؤون المغرب وتنظيمه، وتنقل له مخصصاً سنوياً. لكن معاهدة فيس-مدينة كانت ضد مصالح المغاربة، وسبب ثورة 1912.
relevent_chunks:  ['الفرنسيين غزاو المغرب فـ 1907 بسباب مجموعة ديال العوامل، منها الحرب الأهلية الحفيظية، اللي كانت مدعومة من طرف الأعيان فـ الجنوب اللي كانو فـ مراكش، و رغبة فرنسا باش تسيطر على المداخيل الجمركية فـ الدار البيضاء. الفرنسيين تا بغاو يحطو رجليهم فـ شمال إفريقيا و يوصلو للثروات الطبيعية، بما فيها البترول و الفوسفاط. زيادة على هادشي، الفرنسيين كانو كيشوفو المغرب كتهديد محتمل لهيمنتهم على البحر الأبيض المتوسط، و شافوه كحليف محتمل فـ المنافسة ديالهم مع البريطانيين.', 'عواقب أزمة أكاد

In [ ]:
# Hybrid Search


In [ ]:
# Hyde Architect

In [ ]:
# Get Relevent Documents

In [ ]:
# Get Context and Give LLM Context and Get Response